In [6]:
"""
DS605 Lab 4 - Airbnb Price Prediction
Task 1 & 2: Data cleaning, feature engineering, model training, tuning, evaluation
"""

import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

# -------------------------------------------------------------------
# 0. FIX WORKING DIRECTORY (always run relative to this file's folder)
# -------------------------------------------------------------------
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

os.chdir(BASE_DIR)
os.makedirs("data", exist_ok=True)
os.makedirs("assets", exist_ok=True)
os.makedirs("model", exist_ok=True)

print("Working directory set to:", os.getcwd())
print("Files in data/:", os.listdir("data"))

# -------------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------------
df = pd.read_csv("data/AB_NYC_2019.csv")
print("Raw shape:", df.shape)
print(df.isnull().sum())

# -------------------------------------------------------------------
# 2. CLEANING
# -------------------------------------------------------------------
drop_cols = ["id", "name", "host_id", "host_name", "last_review"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# reviews_per_month is NaN when number_of_reviews == 0 -> fill with 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Drop rows with missing key fields (should be none, but safe)
df = df.dropna(subset=["neighbourhood_group", "room_type", "latitude", "longitude"])

# -------------------------------------------------------------------
# 3. OUTLIER HANDLING
# -------------------------------------------------------------------
df = df[df["price"] > 0]
upper_limit = df["price"].quantile(0.99)
df = df[df["price"] <= upper_limit]

df["minimum_nights"] = df["minimum_nights"].clip(upper=365)

print("Cleaned shape:", df.shape)

# -------------------------------------------------------------------
# 4. FEATURE ENGINEERING
# -------------------------------------------------------------------
neigh_freq = df["neighbourhood"].value_counts(normalize=True)
df["neighbourhood_freq"] = df["neighbourhood"].map(neigh_freq)
df = df.drop(columns=["neighbourhood"])

df["log_price"] = np.log1p(df["price"])

# -------------------------------------------------------------------
# 5. EDA PLOTS
# -------------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Price Distribution (after outlier removal)")
plt.savefig("assets/price_distribution.png", bbox_inches="tight")
plt.close()

plt.figure(figsize=(6, 4))
sns.boxplot(x="room_type", y="price", data=df)
plt.title("Price by Room Type")
plt.savefig("assets/price_by_room_type.png", bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 6))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.savefig("assets/correlation_heatmap.png", bbox_inches="tight")
plt.close()

# -------------------------------------------------------------------
# 6. FEATURE / TARGET SPLIT
# -------------------------------------------------------------------
target = "log_price"
features = [
    "neighbourhood_group", "room_type", "latitude", "longitude",
    "minimum_nights", "number_of_reviews", "reviews_per_month",
    "calculated_host_listings_count", "availability_365",
    "neighbourhood_freq"
]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

categorical_features = ["neighbourhood_group", "room_type"]
numeric_features = [c for c in features if c not in categorical_features]

# -------------------------------------------------------------------
# 7. PREPROCESSING PIPELINE
# -------------------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# -------------------------------------------------------------------
# 8. MODEL COMPARISON
# -------------------------------------------------------------------
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE)
}

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    y_pred_log = pipe.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    train_r2 = r2_score(np.expm1(y_train), np.expm1(pipe.predict(X_train)))

    results.append({
        "Model": name, "MAE": mae, "RMSE": rmse,
        "Test_R2": r2, "Train_R2": train_r2
    })
    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.3f}, Train_R2={train_r2:.3f}")

results_df = pd.DataFrame(results).sort_values("RMSE")
print("\nModel comparison:\n", results_df)
results_df.to_csv("assets/model_comparison.csv", index=False)

# -------------------------------------------------------------------
# 9. HYPERPARAMETER TUNING (RandomForest, regularized to reduce overfitting)
# -------------------------------------------------------------------
param_dist = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [5, 8, 10, 12, 15],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [5, 10, 20, 30],
    "model__max_features": ["sqrt", "log2", 0.5]
}

best_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

search = RandomizedSearchCV(
    best_pipe, param_distributions=param_dist,
    n_iter=20, cv=5, scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)

print("\nBest params:", search.best_params_)

final_model = search.best_estimator_

train_r2_final = r2_score(np.expm1(y_train), np.expm1(final_model.predict(X_train)))
y_pred = np.expm1(final_model.predict(X_test))
y_true = np.expm1(y_test)

final_mae = mean_absolute_error(y_true, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
final_r2 = r2_score(y_true, y_pred)

print(f"\nFINAL MODEL — MAE: {final_mae:.2f}, RMSE: {final_rmse:.2f}, "
      f"Test R2: {final_r2:.3f}, Train R2: {train_r2_final:.3f}")

# -------------------------------------------------------------------
# 10. SAVE PIPELINE (preprocessing + model together)
# -------------------------------------------------------------------
joblib.dump(final_model, "model/price_pipeline.pkl")
print("\nSaved pipeline to model/price_pipeline.pkl")

Working directory set to: C:\Users\hasan pc\OneDrive\Desktop\DS605\202618025_lab-4
Files in data/: ['AB_NYC_2019.csv', 'assets', 'data', 'model', 'requirements.txt', 'train.ipynb']
Raw shape: (48895, 16)
id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64
Cleaned shape: (48410, 11)
LinearRegression: MAE=48.17, RMSE=83.77, R2=0.355, Train_R2=0.342
Ridge: MAE=48.17, RMSE=83.77, R2=0.355, Train_R2=0